In [ ]:
import os, glob
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ============================================================
# Config
# ============================================================
O3_DIR  = "/pool/datos/reanalisis/merra2/monthly/Total_Ozone_Column"
TEM_DIR = "/pool/datos/reanalisis/merra2/daily/zm"

LAT_O3 = (60, 90)     # polar cap for O3
LAT_FZ = (40, 80)     # wave driving lat band
PLEV_FZ = 10000.0     # Pa (100 hPa)

# Seasons
O3_SEASON = "MA"      # Mar-Apr
FZ_SEASON = "DJF"     # Dec-Jan-Feb

# Output
OUT_PNG = "./MERRA2_scatter_FzDJF_vs_O3MA_mean.png"


# ============================================================
# Helpers
# ============================================================
def coslat_weights(lat_da):
    return np.cos(np.deg2rad(lat_da))

def standardize_da(da):
    mu = float(da.mean().values)
    sd = float(da.std().values)
    return (da - mu) / sd

def year_from_winter_label(time_da):
    """
    For QS-DEC resample: DJF season is labeled at Dec (e.g. 1979-12 for DJF 1980).
    We want winter-year = Dec.year + 1.
    """
    return (time_da.dt.year + 1).astype(int)


# ============================================================
# 1) O3: build yearly MA mean (polar cap 60-90N)
# ============================================================
def build_o3_ma_mean_pcap():
    files = sorted(glob.glob(os.path.join(O3_DIR, "MERRA2_*.tavgM_2d_slv_Nx.*.SUB.nc")))
    if not files:
        raise FileNotFoundError(f"No O3 files found in {O3_DIR}")

    def _pre(ds):
        keep_vars = []
        if "TO3" in ds:
            keep_vars.append("TO3")
        return ds[keep_vars] if keep_vars else ds

    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        parallel=False,
        preprocess=_pre
    )

    if "TO3" not in ds:
        raise KeyError("TO3 not found in O3 monthly files")

    to3 = ds["TO3"]  # (time,lat,lon)

    # polar cap mean (60-90N) with cos(lat) weights
    to3_cap = to3.where((to3["lat"] >= LAT_O3[0]) & (to3["lat"] <= LAT_O3[1]), drop=True)
    w = coslat_weights(to3_cap["lat"])
    to3_cap_mean = to3_cap.weighted(w).mean("lat").mean("lon")  # (time,)

    # MA mean each year
    ma = to3_cap_mean.where(to3_cap_mean["time"].dt.month.isin([3, 4]), drop=True)
    o3_ma = ma.groupby("time.year").mean("time")  # (year,)

    o3_ma.name = "O3_MA_mean_60_90N"
    o3_ma.attrs["units"] = to3.attrs.get("units", "Dobsons")

    ds.close()
    return o3_ma


# ============================================================
# 2) Fz: build yearly DJF mean from daily epfz (100 hPa, 40-80N weighted mean)
# ============================================================
def build_fz_djf_mean_epfz_100hpa():
    files = sorted(glob.glob(os.path.join(TEM_DIR, "MERRA2_day_epfz_*.nc")))
    if not files:
        raise FileNotFoundError(f"No epfz files found in {TEM_DIR}")

    def _pre(ds):
        keep = []
        if "epfz" in ds:
            keep.append("epfz")
        return ds[keep] if keep else ds

    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        parallel=False,
        preprocess=_pre
    )

    if "epfz" not in ds:
        raise KeyError("epfz not found in daily TEM files")

    epfz = ds["epfz"]  # (time,plev,lat,lon) with lon=1

    # select 100 hPa
    epfz_100 = epfz.sel(plev=PLEV_FZ, method="nearest")

    # drop lon dimension if lon=1
    if "lon" in epfz_100.dims and epfz_100.sizes["lon"] == 1:
        epfz_100 = epfz_100.isel(lon=0, drop=True)

    # lat 40-80N (order independent)
    epfz_band = epfz_100.where((epfz_100["lat"] >= LAT_FZ[0]) & (epfz_100["lat"] <= LAT_FZ[1]), drop=True)

    # coslat-weighted mean over lat -> daily series
    w = coslat_weights(epfz_band["lat"])
    fz_daily = epfz_band.weighted(w).mean("lat")  # (time,)

    # ---- DJF mean by winter-year ----
    seas = fz_daily.resample(time="QS-DEC").mean()
    djf = seas.where(seas["time"].dt.month == 12, drop=True)  # Dec-labeled seasons

    winter_year = year_from_winter_label(djf["time"]).data
    fz_djf = djf.assign_coords(year=("time", winter_year)).swap_dims({"time": "year"}).drop_vars("time")

    fz_djf.name = "Fz_DJF_mean_epfz_100hPa_40_80N"
    fz_djf.attrs["units"] = epfz.attrs.get("units", "m3 s-2")

    ds.close()
    return fz_djf


# ============================================================
# 3) Plot: standardized scatter (common years) + highlight lowest 3 O3 years
# ============================================================
def plot_scatter_merra2():
    o3_ma = build_o3_ma_mean_pcap()          # (year,)
    fz_djf = build_fz_djf_mean_epfz_100hpa() # (year,)

    # common years intersection
    common_years = np.intersect1d(o3_ma["year"].values.astype(int), fz_djf["year"].values.astype(int))
    common_years = np.sort(common_years)

    print("O3 year range:", int(o3_ma.year.min()), "to", int(o3_ma.year.max()), "N=", int(o3_ma.size))
    print("Fz year range:", int(fz_djf.year.min()), "to", int(fz_djf.year.max()), "N=", int(fz_djf.size))
    print("Common years:", int(common_years.min()), "to", int(common_years.max()), "N=", int(common_years.size))
    print("O3 units:", o3_ma.attrs.get("units", "(missing)"))
    print("Fz units:", fz_djf.attrs.get("units", "(missing)"))

    x_raw = fz_djf.sel(year=common_years)
    y_raw = o3_ma.sel(year=common_years)

    # sanity check
    nfx = int(np.isfinite(x_raw.values).sum())
    nfy = int(np.isfinite(y_raw.values).sum())
    print(f"[fz_x (raw)] size={x_raw.size}, finite={nfx}")
    print(f"[o3_y (raw)] size={y_raw.size}, finite={nfy}")

    # drop non-finite pairs
    mask = np.isfinite(x_raw.values) & np.isfinite(y_raw.values)
    years_used = common_years[mask]
    x_raw = x_raw.sel(year=years_used)
    y_raw = y_raw.sel(year=years_used)

    # -------- NEW: find lowest 3 years by MA O3 (raw, not standardized) --------
    # Use raw y values to rank; then highlight their standardized coordinates on the plot.
    y_vals = y_raw.values
    if y_vals.size < 3:
        raise ValueError(f"Not enough years to pick 3 minima: N={y_vals.size}")

    idx3 = np.argsort(y_vals)[:3]              # indices of smallest 3 O3 values
    years_low3 = years_used[idx3].astype(int)  # corresponding years (MA year)
    # sort for nicer legend order (lowest -> higher)
    years_low3 = years_low3[np.argsort(y_raw.sel(year=years_low3).values)]

    print("Lowest 3 MA O3 years (raw):", years_low3.tolist())
    for yy in years_low3:
        print(f"  year={yy}, O3(MA)={float(y_raw.sel(year=yy).values):.3f} {o3_ma.attrs.get('units','')}")


    # standardize (z-score) over the used period
    x = standardize_da(x_raw)
    y = standardize_da(y_raw)

    # stats
    r, p = pearsonr(x.values, y.values)

    # build masks for plotting
    is_low3 = np.isin(years_used.astype(int), years_low3)
    years_other = years_used[~is_low3]
    years_low = years_used[is_low3]

    # plot
    plt.figure(figsize=(8.2, 6.2), dpi=140)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.axvline(0, color="k", lw=0.8, alpha=0.5)
    plt.xlim(-5, 5)
    plt.ylim(-5, 5)

    # ---- base scatter (others) in grey ----
    plt.scatter(
        x.sel(year=years_other).values,
        y.sel(year=years_other).values,
        s=55, alpha=0.55, edgecolors="none",
        color="0.55",
        label=f"Other years (N={len(years_other)})"
    )

    # ---- highlight lowest 3 with colors + legend ----
    highlight_colors = ["tab:red", "tab:blue", "tab:green"]
    for c, yy in zip(highlight_colors, years_low3):
        plt.scatter(
            float(x.sel(year=yy).values),
            float(y.sel(year=yy).values),
            s=90, alpha=0.95, edgecolors="k", linewidths=0.4,
            color=c,
            label=f"Lowest MA O3: {int(yy)}"
        )

    info = f"Years: {int(years_used.min())}–{int(years_used.max())}\nN = {len(years_used)}\nr = {r:.3f}\np = {p:.2e}"
    plt.text(
        0.03, 0.97, info,
        transform=plt.gca().transAxes,
        va="top", ha="left",
        fontsize=10,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.80, edgecolor="none")
    )

    plt.xlabel("DJF EP flux (epfz) at 100 hPa, 40–80°N (standardized)", fontsize=11)
    plt.ylabel("MA total ozone column (TO3) 60–90°N (standardized)", fontsize=11)
    plt.title("MERRA2: DJF EP flux vs MA Total Ozone (z-scored)", loc="left", fontweight="bold")

    plt.legend(frameon=True, fontsize=9, loc="best")
    plt.tight_layout()
    plt.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", OUT_PNG)


if __name__ == "__main__":
    plot_scatter_merra2()

In [ ]:
# ============================================================
# DEBUG BLOCK for MERRA2 O3 / EP metadata & sanity checks
# 不写正式输出文件，只做诊断
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
import xarray as xr

# ---------------- Paths ----------------
O3_DIR = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/O3"
EP_NC  = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/EPflux_daily/MERRA2_EPFLUX_Daily_Series_1980_2026_noW.nc"

LAT_BAND_O3 = (60, 90)
PLEV_TOP = 30.0
PLEV_BOT = 70.0

# 只抽样看一个 O3 文件，避免太重
# 想换年份就改 index
O3_FILE_INDEX = 0


# ============================================================
# Helpers
# ============================================================
def print_header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def sel_latband(da, lat0=60.0, lat1=90.0, lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    return da.sel({lat_name: slc})

def summarize_time_axis(time_values, name="time"):
    print_header(f"{name} axis summary")
    t = pd.to_datetime(time_values)
    print(f"Length                : {len(t)}")
    print(f"First                 : {t[0]}")
    print(f"Last                  : {t[-1]}")
    if len(t) > 1:
        dt_hours = np.diff(t.values).astype("timedelta64[s]").astype(float) / 3600.0
        uniq, cnt = np.unique(np.round(dt_hours, 6), return_counts=True)
        print("Unique time step hours:")
        for u, c in zip(uniq[:10], cnt[:10]):
            print(f"  {u:g} h  -> {c} times")
        if len(uniq) > 10:
            print(f"  ... and {len(uniq)-10} more")
    unique_days = len(pd.Index(t.normalize()).unique())
    print(f"Unique calendar days  : {unique_days}")
    print(f"Samples / unique days : {len(t) / unique_days:.3f}")

def summarize_coord(da, coord_name):
    print_header(f"Coordinate summary: {coord_name}")
    coord = da[coord_name]
    vals = np.asarray(coord.values)
    print(f"dtype   : {vals.dtype}")
    print(f"shape   : {vals.shape}")
    print(f"attrs   : {dict(coord.attrs)}")
    if vals.ndim == 1 and len(vals) > 0:
        print(f"first 10: {vals[:10]}")
        print(f"last 10 : {vals[-10:]}")
        try:
            diffs = np.diff(vals.astype(float))
            if len(diffs) > 0:
                print(f"monotonic increasing? {np.all(diffs > 0)}")
                print(f"monotonic decreasing? {np.all(diffs < 0)}")
        except Exception:
            pass

def detect_o3_mode_from_attrs(da_o3):
    txt = " ".join([
        str(da_o3.attrs.get("units", "")),
        str(da_o3.attrs.get("long_name", "")),
        str(da_o3.attrs.get("standard_name", "")),
    ]).lower()

    if "ppbv" in txt:
        return "ppbv"
    if "ppmv" in txt:
        return "ppmv"
    if ("mass mixing ratio" in txt) or ("kg/kg" in txt) or ("kg kg-1" in txt) or ("kg kg^-1" in txt):
        return "mass_mixing_ratio"
    if ("mole fraction" in txt) or ("mol/mol" in txt) or ("mol mol-1" in txt) or ("vmr" in txt) or ("volume mixing ratio" in txt):
        return "mole_fraction"
    return "unclear"

def pressure_layer_edges_from_centers(lev_hpa):
    lev = np.asarray(lev_hpa, dtype=float)
    lev_sorted = np.sort(lev)  # ascending: top -> bottom
    logp = np.log(lev_sorted)
    mids = np.exp(0.5 * (logp[:-1] + logp[1:]))
    edges = np.empty(lev_sorted.size + 1, dtype=float)
    edges[1:-1] = mids
    edges[0] = lev_sorted[0] ** 2 / mids[0]
    edges[-1] = lev_sorted[-1] ** 2 / mids[-1]
    return lev_sorted, edges

def to_mole_fraction(da, mode):
    M_AIR = 28.964e-3
    M_O3  = 47.9982e-3

    if mode == "mass_mixing_ratio":
        return da * (M_AIR / M_O3)
    elif mode == "mole_fraction":
        return da
    elif mode == "ppmv":
        return da / 1e6
    elif mode == "ppbv":
        return da / 1e9
    else:
        raise ValueError(mode)

def quick_partial_column_du(da_o3_3d, p_top_hpa=30.0, p_bot_hpa=70.0, mode="mass_mixing_ratio"):
    """
    da_o3_3d dims expected: (lev, lat, lon) or compatible
    Returns polar-mean DU over chosen layer
    """
    g      = 9.80665
    M_air  = 28.964e-3
    Na     = 6.02214e23
    DU_den = 2.687e20
    factor = Na / (g * M_air * DU_den)

    da = da_o3_3d.sortby("lev")
    lev_sorted, edges_hpa = pressure_layer_edges_from_centers(da["lev"].values)
    da = da.sel(lev=lev_sorted)

    da = to_mole_fraction(da, mode=mode)

    p_layer_top_pa = xr.DataArray(edges_hpa[:-1] * 100.0, coords={"lev": da["lev"]}, dims=("lev",))
    p_layer_bot_pa = xr.DataArray(edges_hpa[1:]  * 100.0, coords={"lev": da["lev"]}, dims=("lev",))

    pT = p_top_hpa * 100.0
    pB = p_bot_hpa * 100.0

    upper = xr.where(p_layer_top_pa > pT, p_layer_top_pa, pT)
    lower = xr.where(p_layer_bot_pa < pB, p_layer_bot_pa, pB)
    overlap = xr.where(lower > upper, lower - upper, 0.0)

    col = (da * overlap * factor).sum("lev")

    # 60-90N cos(lat) weighted mean + lon mean
    col = sel_latband(col, LAT_BAND_O3[0], LAT_BAND_O3[1], "lat")
    w = np.cos(np.deg2rad(col["lat"]))
    ts = col.weighted(w).mean(("lat", "lon"))
    return float(ts.values)

def print_data_range(name, da):
    vals = np.asarray(da.values)
    vals = vals[np.isfinite(vals)]
    print_header(f"{name} value range")
    if vals.size == 0:
        print("All values are NaN.")
        return
    qs = np.nanpercentile(vals, [0, 1, 5, 25, 50, 75, 95, 99, 100])
    labs = ["min", "p01", "p05", "p25", "p50", "p75", "p95", "p99", "max"]
    for lab, q in zip(labs, qs):
        print(f"{lab:>4s}: {q:.6e}")

def guess_plev_unit(plev_vals):
    vmax = np.nanmax(np.abs(plev_vals))
    if vmax > 2000:
        return "Pa (likely)", 10000.0
    return "hPa (likely)", 100.0


# ============================================================
# 1) Find files
# ============================================================
print_header("File discovery")

o3_files = sorted(glob.glob(os.path.join(O3_DIR, "MERRA2.O3.*.nc")))
print(f"O3 yearly files found: {len(o3_files)}")
if len(o3_files) == 0:
    raise FileNotFoundError(f"No O3 files found in {O3_DIR}")

for f in o3_files[:5]:
    print("  ", os.path.basename(f))
if len(o3_files) > 5:
    print("  ...")
    for f in o3_files[-3:]:
        print("  ", os.path.basename(f))

o3_file = o3_files[O3_FILE_INDEX]
print(f"\nSelected O3 debug file: {o3_file}")
print(f"EP debug file         : {EP_NC}")


# ============================================================
# 2) Inspect one O3 file
# ============================================================
print_header("Open one O3 file")
with xr.open_dataset(o3_file) as ds_o3:
    print(ds_o3)

    if "O3" not in ds_o3:
        raise KeyError("Variable 'O3' not found in selected O3 file.")

    da_o3 = ds_o3["O3"]

    print_header("O3 attrs")
    print(dict(da_o3.attrs))

    print_header("Dataset attrs")
    print(dict(ds_o3.attrs))

    print_header("O3 dims / shape")
    print(f"dims  : {da_o3.dims}")
    print(f"shape : {da_o3.shape}")

    summarize_coord(da_o3, "time")
    summarize_coord(da_o3, "lev")
    summarize_coord(da_o3, "lat")
    if "lon" in da_o3.coords:
        summarize_coord(da_o3, "lon")

    summarize_time_axis(da_o3.time.values, name="O3 time")

    print_header("O3 unit guess from attrs")
    unit_guess = detect_o3_mode_from_attrs(da_o3)
    print(f"Detected by attrs: {unit_guess}")

    # 看几个层的值量级
    print_data_range("O3 full field", da_o3)

    # 抽样看 30-70 hPa、60-90N、第一天
    da_sample = da_o3.isel(time=0)
    print_data_range("O3 sample day full 3D", da_sample)

    da_3070 = da_sample.sel(lev=slice(PLEV_TOP, PLEV_BOT))
    print_data_range(f"O3 sample day {PLEV_TOP}-{PLEV_BOT} hPa", da_3070)

    # 再看极区子域
    da_polar = sel_latband(da_3070, LAT_BAND_O3[0], LAT_BAND_O3[1], "lat")
    print_data_range(f"O3 sample day {PLEV_TOP}-{PLEV_BOT} hPa, {LAT_BAND_O3[0]}-{LAT_BAND_O3[1]}N", da_polar)

    # 尝试不同单位假设下算一个 partial column
    print_header("Quick partial-column test (sample first day, 60-90N, 30-70 hPa)")
    test_day = sel_latband(da_o3.isel(time=0), LAT_BAND_O3[0], LAT_BAND_O3[1], "lat")

    for mode in ["mass_mixing_ratio", "mole_fraction", "ppmv", "ppbv"]:
        try:
            val = quick_partial_column_du(test_day, p_top_hpa=PLEV_TOP, p_bot_hpa=PLEV_BOT, mode=mode)
            print(f"{mode:>20s} -> {val:10.3f} DU")
        except Exception as e:
            print(f"{mode:>20s} -> FAILED: {e}")


# ============================================================
# 3) Inspect EP file
# ============================================================
print_header("Open EP file")
with xr.open_dataset(EP_NC) as ds_ep:
    print(ds_ep)

    var_fz = "Fz" if "Fz" in ds_ep.data_vars else ("EP2_cart" if "EP2_cart" in ds_ep.data_vars else None)
    if var_fz is None:
        raise KeyError("Neither 'Fz' nor 'EP2_cart' found in EP file.")

    da_fz = ds_ep[var_fz]

    print_header(f"{var_fz} attrs")
    print(dict(da_fz.attrs))

    print_header(f"{var_fz} dims / shape")
    print(f"dims  : {da_fz.dims}")
    print(f"shape : {da_fz.shape}")

    if "time" in da_fz.coords:
        summarize_coord(da_fz, "time")
        summarize_time_axis(da_fz.time.values, name=f"{var_fz} time")

    if "plev" in da_fz.coords:
        summarize_coord(da_fz, "plev")
        plev_vals = np.asarray(da_fz["plev"].values, dtype=float)
        unit_guess, target_100hpa = guess_plev_unit(plev_vals)
        print_header("plev unit guess")
        print(f"plev looks like: {unit_guess}")
        print(f"100 hPa target should be selected with: {target_100hpa}")

        try:
            da_100 = da_fz.sel(plev=target_100hpa, method="nearest")
            print_header("Selected 100 hPa level")
            print(da_100)
            print_data_range(f"{var_fz} at 100 hPa", da_100)
        except Exception as e:
            print(f"Selecting 100 hPa level failed: {e}")

    if "lat" in da_fz.coords:
        summarize_coord(da_fz, "lat")

    print_data_range(f"{var_fz} full field", da_fz)


# ============================================================
# 4) Simple interpretation hints
# ============================================================
print_header("How to interpret the debug output")
print(
    "1) 如果 O3 attrs 直接写了 'mass mixing ratio' / 'kg/kg'，那就按 mass_mixing_ratio 处理。\n"
    "2) 如果 O3 attrs 写了 'mole fraction' / 'vmr' / 'mol/mol'，那就按 mole_fraction。\n"
    "3) 如果 attrs 不清楚，就看上面 sample day 的 quick partial-column test：\n"
    "   哪一种假设给出的 30-70 hPa polar-cap O3 更像合理数量级，就优先用哪种。\n"
    "4) 对 EP 文件，如果 plev 最大值像 1000、100、70 这种，就是 hPa；\n"
    "   如果像 100000、10000 这种，就是 Pa。\n"
    "5) 如果 time summary 里 Samples/unique days = 1.000，说明就是 daily。\n"
)